In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

In [ ]:
cols = ["ID","cThick","UCSize", "UCShape", "Adhesion", "CECSize", "Bare", "Bland", "Normal", "Mitoses","class"]
df = pd.read_csv("breast-cancer-wisconsin.data", names=cols)
df.drop("Bare", inplace=True, axis=1)
df.head()

In [ ]:
train, valid, test = np.split(df.sample(frac=1), [int(.6*len(df)), int(.8*len(df))])

In [ ]:
def scale_dataset(dataframe, oversample=False):
    x = dataframe[dataframe.columns[:-1]].values
    y = dataframe[dataframe.columns[-1]].values

    scaler = StandardScaler()
    if oversample:
        ros = RandomOverSampler()
        x, y = ros.fit_resample(x, y)
        
    x = scaler.fit_transform(x)
    data = np.hstack((x,np.reshape(y, (-1,1))))
    return data, x, y

In [ ]:
train, X_train, Y_train = scale_dataset(train, oversample=True)
valid, X_valid, Y_valid = scale_dataset(valid, oversample=False)
test, X_test, Y_test = scale_dataset(test, oversample=False)

# Importing ML algorithms and Classification Report

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# kNN

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train, Y_train)
Y_pred = knn_model.predict(X_test)
print(classification_report(Y_test, Y_pred))

# Naive Bayes

In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train, Y_train)
Y_pred = nb_model.predict(X_test)
print(classification_report(Y_test, Y_pred))

# Logistic Regression

In [ ]:
lr_model = LogisticRegression()
lr_model.fit(X_train, Y_train)
Y_pred = lr_model.predict(X_test)
print(classification_report(Y_test, Y_pred))

# SVC

In [ ]:
svc_model = SVC()
svc_model.fit(X_train, Y_train)
Y_pred = nb_model.predict(X_test)
print(classification_report(Y_test, Y_pred))

# neural network

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

def scale_dataset(dataframe, oversample=False):
    x = dataframe[dataframe.columns[:-1]].values
    y = dataframe[dataframe.columns[-1]].values

    scaler = StandardScaler()
    if oversample:
        ros = RandomOverSampler()
        x, y = ros.fit_resample(x, y)
        
    x = scaler.fit_transform(x)
    data = np.hstack((x,np.reshape(y, (-1,1))))
    return data, x, y

cols = ["ID","cThick","UCSize", "UCShape", "Adhesion", "CECSize", "Bare", "Bland", "Normal", "Mitoses","class"]
df = pd.read_csv("breast-cancer-wisconsin.data", names=cols)
df.drop("Bare", inplace=True, axis=1)
df.head()

df["class"] = df["class"].map({2: 0, 4: 1})

train, valid, test = np.split(df.sample(frac=1), [int(.6*len(df)), int(.8*len(df))])

train, X_train, Y_train = scale_dataset(train, oversample=True)
valid, X_valid, Y_valid = scale_dataset(valid, oversample=False)
test, X_test, Y_test = scale_dataset(test, oversample=False)

C:\Users\Drew Miccoli\Lib\site-packages\numpy\core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [85]:
import torch
from torch import nn

# 1. Load Data
torch.set_default_tensor_type(torch.DoubleTensor)

# Get cpu, gpu or mps device for training.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device using {device}")

# 2. Define the Model
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(9, 10),
            nn.ReLU(),
            nn.Linear(10,1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.linear_relu_stack(x)
        return x

model = NeuralNetwork().to(device)

# 3. Define Loss Function and Optimizer
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 4. The Training Loop Function
def train(model, loss_fn, optimizer,epochs):
    model.train() # set model to training mode
    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        correct = 0
        for idx,x in enumerate(X_train):
            X, Y = torch.tensor(x).to(device), torch.tensor([Y_train[idx]]).to(device).to(torch.float64)
    
            # Forward pass: Compute prediction and loss
            pred = model(X)
            loss = loss_fn(pred, Y)
    
            # Backward pass: Compute gradients and update weights
            optimizer.zero_grad() # clear previous gradients
            loss.backward() # backpropagate loss
            optimizer.step() # update weights
    
            # accuracy
            if pred[0] > .5:
                pred[0] = 1
            else:
                pred[0] = 0
            correct += (pred == Y.int()).sum().item()
    
            # loss
        loss = loss.item()
        print(f"loss: {loss:>7f}")
        print(correct/len(X_train))
    print("Done!")

# 5. Run Training
train(model, loss_fn, optimizer,epochs=20)

device using cuda
Epoch 1
-------------------------------
loss: 0.141198
0.7562724014336918
Epoch 2
-------------------------------
loss: 0.019603
0.942652329749104
Epoch 3
-------------------------------
loss: 0.007732
0.9516129032258065
Epoch 4
-------------------------------
loss: 0.004903
0.9498207885304659
Epoch 5
-------------------------------
loss: 0.004057
0.9605734767025089
Epoch 6
-------------------------------
loss: 0.003705
0.9659498207885304
Epoch 7
-------------------------------
loss: 0.003591
0.9659498207885304
Epoch 8
-------------------------------
loss: 0.003589
0.9659498207885304
Epoch 9
-------------------------------
loss: 0.003616
0.9659498207885304
Epoch 10
-------------------------------
loss: 0.003654
0.9659498207885304
Epoch 11
-------------------------------
loss: 0.003714
0.9659498207885304
Epoch 12
-------------------------------
loss: 0.003743
0.967741935483871
Epoch 13
-------------------------------
loss: 0.003648
0.9695340501792115
Epoch 14
---------

In [86]:
def model_eval(model,X_data,Y_data):
    model.eval()
    with torch.no_grad():
        correct = 0
        for idx,x in enumerate(X_data):
            X, Y = torch.tensor(x).to(device), torch.tensor([Y_data[idx]]).to(device).to(torch.float64)
    
            # Forward pass: Compute prediction and loss
            pred = model(X)
            # accuracy
            if pred[0] > .5:
                pred[0] = 1
            else:
                pred[0] = 0
            correct += (pred == Y.int()).sum().item()
    
        print(f"valid accuracy: {correct/len(X_data)}")

In [87]:
model_eval(model,X_valid,Y_valid)

valid accuracy: 0.9714285714285714


In [88]:
model_eval(model,X_test,Y_test)

valid accuracy: 0.9142857142857143
